In [1]:
import os
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt

## Food101 Data Ingestion

Load the [Food101 dataset](https://huggingface.co/datasets/ethz/food101) from HuggingFace — 101,000 images across 101 food categories, split into train (75,750) and validation (25,250).

In [ ]:
# Load train and validation splits from HuggingFace
# dataset = load_dataset("ethz/food101")

# train_ds = dataset["train"]
# val_ds   = dataset["validation"]

# print(f"Train samples : {len(train_ds):,}")
# print(f"Val   samples : {len(val_ds):,}")
# print(f"Features      : {train_ds.features}")

In [2]:
# (Optional) Save dataset to disk for faster reloads in subsequent sessions
SAVE_DIR = "./food101_cache"
# dataset.save_to_disk(SAVE_DIR)
# print(f"Dataset saved to {os.path.abspath(SAVE_DIR)}")

# To reload later:
from datasets import load_from_disk
dataset = load_from_disk(SAVE_DIR)
train_ds = dataset["train"]
val_ds   = dataset["validation"]

In [ ]:
# Class labels
label_names = train_ds.features["label"].names
num_classes = len(label_names)
print(f"Number of classes: {num_classes}")
print(f"First 10 classes : {label_names[:10]}")

In [ ]:
# Visualise a sample grid (one image per class, first 12 classes)
sample = train_ds[0]  # a single dataset item: {'image': <PIL.Image>, 'label': <int>}

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

# Build a quick label->index map for the first sample of each class
label_to_idx = {}
for i, sample in enumerate(train_ds):
    lbl = sample["label"]
    if lbl not in label_to_idx:
        label_to_idx[lbl] = i
    if len(label_to_idx) == 12:
        break

for ax, (lbl, idx) in zip(axes, sorted(label_to_idx.items())):
    img = train_ds[idx]["image"]
    ax.imshow(img)
    ax.set_title(label_names[lbl], fontsize=9)
    ax.axis("off")

plt.suptitle("Food101 — sample images", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# List available models for this API key
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)


In [ ]:
import json
import time
import os
from groq import Groq

print('\n'.join(label_names))

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def get_food_attributes(food_name: str) -> list:
    prompt = (
        f"List 3 to 5 short visual rules or attributes that distinguish "
        f"'{food_name.replace('_', ' ')}' from other foods. "
        f"Focus on appearance, texture, colour, and ingredients visible in a photo. "
        f'Respond with a JSON array of short strings only, no explanation. '
        f'Example format: ["has red tomato sauce", "round flat bread base", "melted cheese on top"]'
    )
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0.3,
    )
    text = response.choices[0].message.content
    start, end = text.find("["), text.rfind("]") + 1
    return json.loads(text[start:end]) if start != -1 else [text.strip()]

food_attributes = {}
for i, cls in enumerate(label_names):
    try:
        food_attributes[cls] = get_food_attributes(cls)
        print(f"[{i+1:>3}/101] {cls:30s} -> {food_attributes[cls]}")
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"[{i+1:>3}/101] {cls:30s} -> ERROR: {type(e).__name__}: {e}")
        food_attributes[cls] = []
    time.sleep(3)

with open("food101_attributes.json", "w") as f:
    json.dump(food_attributes, f, indent=2)

print("\nSaved to food101_attributes.json")


In [ ]:
import pandas as pd

with open("food101_attributes.json") as f:
    food_attributes = json.load(f)

rows = []
for cls, rules in food_attributes.items():
    for i, rule in enumerate(rules, 1):
        rows.append({"class": cls, "rule_no": i, "rule": rule})

df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
df

In [ ]:
import json
import pandas as pd

with open("food101_attributes.json") as f:
    food_attributes = json.load(f)

# --- 1. Build global concept list (deduplicated, lowercased) ---
concept_set = {}  # concept_str -> canonical index
for cls, rules in food_attributes.items():
    for rule in rules:
        key = rule.strip().lower()
        if key not in concept_set:
            concept_set[key] = len(concept_set)

concept_list = list(concept_set.keys())  # ordered list of all unique concepts
print(f"Total unique concepts: {len(concept_list)}")

# --- 2. Build class -> concept binary matrix ---
class_names = list(food_attributes.keys())  # 101 classes in label order
n_classes   = len(class_names)
n_concepts  = len(concept_list)

import numpy as np
concept_matrix = np.zeros((n_classes, n_concepts), dtype=int)

for cls_idx, cls in enumerate(class_names):
    for rule in food_attributes[cls]:
        key = rule.strip().lower()
        concept_matrix[cls_idx, concept_set[key]] = 1

print(f"Concept matrix shape: {concept_matrix.shape}  (classes x concepts)")
print(f"Avg concepts per class: {concept_matrix.sum(axis=1).mean():.1f}")
print(f"Avg classes per concept: {concept_matrix.sum(axis=0).mean():.1f}")

# --- 3. Show as DataFrame ---
df_concepts = pd.DataFrame(concept_matrix, index=class_names, columns=concept_list)
print(df_concepts.iloc[:10, :10])  # preview top-left corner

# --- 4. Save ---
with open("food101_concept_list.json", "w") as f:
    json.dump(concept_list, f, indent=2)

np.save("food101_concept_matrix.npy", concept_matrix)
print("\nSaved food101_concept_list.json and food101_concept_matrix.npy")


In [ ]:
# ── Step 1: Generate global concept vocabulary ──────────────────────────────
import json, os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

prompt = """
You are designing a Concept Bottleneck Model (CBM) for food image classification across 101 classes.

Your task is to generate a vocabulary of 80-100 VISUAL concepts that a human can judge 
as present or absent by looking at a photo alone.

STRICT RULES:
1. Every concept must be directly VISIBLE in an image — no taste, smell, temperature, or cultural knowledge required.
2. Each concept must apply to MULTIPLE food classes (avoid dish-specific concepts like "has pepperoni").
3. Concepts must be DISCRIMINATIVE — they should help distinguish between different foods.
4. Avoid redundancy — do not generate near-duplicate concepts (e.g. "crispy texture" and "crunchy surface").
5. NO ingredient-based concepts unless the ingredient is visually identifiable (e.g. "visible green leaves" yes, "contains beef" no).

Cover these VISUAL dimensions evenly:
- Surface texture     : what does the surface look/feel like visually?
- Color & tone        : dominant colors, gradients, browning, glossiness
- Shape & geometry    : overall form, cross-section shape, aspect ratio
- Internal structure  : visible layers, holes, filling, cross-section if cut
- Cooking appearance  : visual signs of cooking method (char marks, crispy edges, steam-wilted, etc.)
- Sauce & coating     : presence and appearance of liquid, glaze, powder, crust
- Arrangement & context: how is it plated, stacked, wrapped, or garnished?
- Size & portion cues : individual piece vs. shared dish, bite-sized vs. large

Return ONLY a JSON array of short concept strings (2-5 words each).
Do NOT copy the examples below — they are only to illustrate the format and specificity level.
Format: ["char marks visible", "layered cross section", "glossy glaze coating", "bite sized pieces", ...]
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1500,
    temperature=0.3,
)
text = response.choices[0].message.content

# Robust JSON extraction: strip markdown fences if present
import re
fenced = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
if fenced:
    text = fenced.group(1)

start, end = text.find("["), text.rfind("]") + 1
if start == -1:
    raise ValueError(f"No JSON array found in response:\n{text}")
try:
    global_concepts = json.loads(text[start:end])
except json.JSONDecodeError as e:
    print(f"Raw response:\n{text}")
    raise e

# Normalise
global_concepts = [c.strip().lower() for c in global_concepts]
global_concepts = list(dict.fromkeys(global_concepts))  # deduplicate, preserve order

print(f"Global concept vocabulary: {len(global_concepts)} concepts\n")
for i, c in enumerate(global_concepts):
    print(f"  {i:>3}. {c}")

with open("food101_global_concepts.json", "w") as f:
    json.dump(global_concepts, f, indent=2)
print("\nSaved to food101_global_concepts.json")


In [25]:
# ── Step 2: Label each class with the global concept vocabulary ───────────────
import json, time, numpy as np, os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

with open("food101_global_concepts.json") as f:
    global_concepts = json.load(f)

concept_index = {c.strip().lower(): i for i, c in enumerate(global_concepts)}
n_concepts = len(global_concepts)
concept_list_str = json.dumps(global_concepts)

def label_class(food_name: str, max_retries: int = 3) -> list[int]:
    prompt = (
        f"You are a visual annotation expert building training labels for a food vision model.\n\n"
        f"Food class: '{food_name.replace('_', ' ')}'\n\n"
        f"Imagine a TYPICAL photo of this food as it appears in a restaurant or food dataset. "
        f"From the concept list below, select ONLY concepts that would be clearly visible "
        f"in most photos of this food — not just occasionally.\n\n"
        f"Concept list:\n{concept_list_str}\n\n"
        f"Rules:\n"
        f"1. Only include concepts visible in >70% of typical photos of this food.\n"
        f"2. Do NOT include concepts based on taste, smell, or cultural knowledge.\n"
        f"3. Select between 5 and 15 concepts.\n"
        f"4. Return ONLY exact concept strings from the list above as a JSON array.\n"
        f"5. Strings must match the list exactly — no paraphrasing, no new concepts.\n\n"
        f"Example output format: [\"golden brown crust\", \"served in bowl\", \"glossy glaze coating\"]"
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",  # 更大的模型，判断更准确
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1024,  # 足够装下所有 concept
                temperature=0.1,
            )
            text = response.choices[0].message.content.strip()
            start, end = text.find("["), text.rfind("]") + 1
            if start == -1 or end == 0:
                raise ValueError("No JSON array found in response")
            applicable = json.loads(text[start:end])

            # Convert to binary vector，做 lowercase 匹配防止大小写不一致
            vec = [0] * n_concepts
            matched, skipped = [], []
            for c in applicable:
                key = c.strip().lower()
                if key in concept_index:
                    vec[concept_index[key]] = 1
                    matched.append(key)
                else:
                    skipped.append(c)  # 模型返回了不在列表里的 concept

            if skipped:
                print(f"  ⚠️  Skipped {len(skipped)} out-of-list concepts: {skipped[:3]}...")

            return vec

        except Exception as e:
            print(f"  Attempt {attempt+1}/{max_retries} failed: {e}")
            time.sleep(2 ** attempt)  # exponential backoff

    print(f"  ❌ All retries failed for '{food_name}', returning zero vector")
    return [0] * n_concepts


concept_matrix = {}
for i, cls in enumerate(label_names):
    concept_matrix[cls] = label_class(cls)
    n_active = sum(concept_matrix[cls])
    active = [global_concepts[j] for j, v in enumerate(concept_matrix[cls]) if v]
    print(f"[{i+1:>3}/101] {cls:30s} ({n_active:>2} concepts) -> {active}")
    time.sleep(1)  # 70b 模型稍微慢一点，间隔可以短一些

# Save
matrix_np = np.array([concept_matrix[cls] for cls in label_names], dtype=int)
np.save("food101_concept_matrix.npy", matrix_np)
with open("food101_concept_list.json", "w") as f:
    json.dump(global_concepts, f, indent=2)

print(f"\nConcept matrix shape : {matrix_np.shape}")
print(f"Avg concepts per class: {matrix_np.sum(axis=1).mean():.1f}")
print(f"Avg classes per concept: {matrix_np.sum(axis=0).mean():.1f}")
print(f"Zero vectors (failed): {(matrix_np.sum(axis=1) == 0).sum()}")
print("Saved food101_concept_matrix.npy and food101_concept_list.json")


[  1/101] apple_pie                      (10 concepts) -> ['browned edges visible', 'crispy texture evident', 'deeply colored surface', 'flaky crust present', 'golden brown color', 'regular shape', 'shiny surface', 'well-cooked center', 'wrinkled surface', 'yellowish color']
[  2/101] baby_back_ribs                 (12 concepts) -> ['browned edges visible', 'deeply colored surface', 'evenly cooked surface', 'golden brown color', 'reddish brown color', 'regular shape', 'separate pieces stacked', 'shiny surface', 'sticky surface', 'well-cooked center', 'wrinkled surface', 'yellowish color']
[  3/101] baklava                        (14 concepts) -> ['deeply colored surface', 'distinct layers visible', 'flaky crust present', 'golden brown color', 'irregular shape', 'layered cross section', 'multiple layers stacked', 'regular shape', 'separate pieces stacked', 'three-dimensional shape', 'tightly packed pieces', 'well-cooked center', 'wrinkled surface', 'yellowish color']
[  4/101] beef_carp

In [28]:
# ── Fix: re-label the 4 failed (zero-vector) classes ────────────────────────
import json, time, numpy as np, os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

with open("food101_global_concepts.json") as f:
    global_concepts = json.load(f)
concept_index = {c: i for i, c in enumerate(global_concepts)}
n_concepts = len(global_concepts)
concept_list_str = json.dumps(global_concepts)

matrix = np.load("food101_concept_matrix.npy")

# Find zero-vector classes
failed = [(i, cls) for i, cls in enumerate(label_names) if matrix[i].sum() == 0]
print(f"Re-running {len(failed)} failed classes: {[cls for _, cls in failed]}")

def label_class(food_name: str) -> list[int]:
    prompt = (
        f"You are labelling food images for a vision model.\n"
        f"Food class: '{food_name.replace('_', ' ')}'\n"
        f"Global concept list:\n{concept_list_str}\n\n"
        f"Return ONLY a JSON array of the concept strings that visually apply "
        f"to this food class. Choose only concepts clearly visible in a typical photo. "
        f"No explanation, no extra text."
    )
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.1,
    )
    text = response.choices[0].message.content
    import re
    fenced = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if fenced:
        text = fenced.group(1)
    start, end = text.find("["), text.rfind("]") + 1
    applicable = json.loads(text[start:end])
    vec = [0] * n_concepts
    for c in applicable:
        key = c.strip().lower()
        if key in concept_index:
            vec[concept_index[key]] = 1
    return vec

for cls_idx, cls in failed:
    try:
        vec = label_class(cls)
        matrix[cls_idx] = vec
        active = [global_concepts[j] for j, v in enumerate(vec) if v]
        print(f"  {cls:30s} ({len(active)} concepts) -> {active}")
    except Exception as e:
        print(f"  {cls:30s} -> STILL FAILED: {e}")
    time.sleep(3)

np.save("food101_concept_matrix.npy", matrix)
print(f"\nUpdated matrix saved.")
print(f"Zero vectors remaining: {sum(1 for i in range(len(label_names)) if matrix[i].sum() == 0)}")
print(f"Avg concepts per class : {matrix.sum(axis=1).mean():.1f}")
print(f"Avg classes per concept: {matrix.sum(axis=0).mean():.1f}")


Re-running 4 failed classes: ['churros', 'cup_cakes', 'panna_cotta', 'poutine']
  churros                        (15 concepts) -> ['browned edges visible', 'crispy texture evident', 'deeply colored surface', 'distinct layers visible', 'evenly cooked surface', 'glossy glaze coating', 'golden brown color', 'irregular shape', 'layered cross section', 'lightly browned surface', 'regular shape', 'shiny surface', 'swirled pattern', 'three-dimensional shape', 'well-cooked center']
  cup_cakes                      (15 concepts) -> ['browned edges visible', 'crispy texture evident', 'deeply colored surface', 'distinct layers visible', 'evenly cooked surface', 'glossy glaze coating', 'golden brown color', 'irregular shape', 'layered cross section', 'lightly browned surface', 'regular shape', 'shiny surface', 'swirled pattern', 'three-dimensional shape', 'well-cooked center']
  panna_cotta                    (33 concepts) -> ['browned edges visible', 'evenly cooked surface', 'glossy glaze coating

In [29]:
import json, numpy as np, pandas as pd

with open("food101_global_concepts.json") as f:
    global_concepts = json.load(f)
matrix = np.load("food101_concept_matrix.npy")

print(f"Matrix shape         : {matrix.shape}  (expected 101 x {len(global_concepts)})")
print(f"Avg concepts/class   : {matrix.sum(axis=1).mean():.1f}")
print(f"Avg classes/concept  : {matrix.sum(axis=0).mean():.1f}")
print(f"Zero-vector classes  : {(matrix.sum(axis=1) == 0).sum()}")
print(f"Unused concepts      : {(matrix.sum(axis=0) == 0).sum()}")
print()

# Per-class summary
rows = [{"class": label_names[i], "n_concepts": int(matrix[i].sum())} for i in range(len(label_names))]
df = pd.DataFrame(rows)
print("Classes with 0 concepts:")
print(df[df.n_concepts == 0])
print()
print("Classes with fewest concepts (bottom 10):")
print(df.nsmallest(10, "n_concepts").to_string(index=False))
print()
print("Classes with most concepts (top 10):")
print(df.nlargest(10, "n_concepts").to_string(index=False))


Matrix shape         : (101, 92)  (expected 101 x 92)
Avg concepts/class   : 11.9
Avg classes/concept  : 13.1
Zero-vector classes  : 0
Unused concepts      : 21

Classes with 0 concepts:
Empty DataFrame
Columns: [class, n_concepts]
Index: []

Classes with fewest concepts (bottom 10):
               class  n_concepts
    chocolate_mousse           6
              hummus           7
              samosa           7
strawberry_shortcake           7
            tiramisu           7
             ceviche           8
        french_fries           8
          cheesecake           9
        deviled_eggs           9
             falafel           9

Classes with most concepts (top 10):
      class  n_concepts
panna_cotta          33
   pad_thai          18
    poutine          18
    ravioli          16
      sushi          16
carrot_cake          15
    churros          15
  cup_cakes          15
  escargots          15
greek_salad          15
